# Fake Drug Text/Barcode Checker — NextGen Capstone

**Important:** This notebook is an educational ML prototype. The dataset is synthetic and the model must not be used to determine whether a real medicine is genuine or counterfeit.


## 1. Problem
Counterfeit and falsified medicines are a public-health concern. This capstone explores whether machine learning can screen structured product information for patterns that deserve further verification.

The prototype uses synthetic records labelled `genuine_like` and `suspicious`. A real deployment would require a validated, regulator-approved dataset and a carefully designed verification workflow.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import joblib


In [ ]:
df = pd.read_csv('../data/synthetic_drug_records.csv')
df.head()


In [ ]:
print(df.shape)
print(df['label'].value_counts())
df.isna().sum()


## 2. Prepare text features
We combine the product fields into one text representation. Word and character TF-IDF features help the classifier learn both words and identifier patterns.


In [ ]:
feature_cols = ['drug_name','nafdac_number','manufacturer','batch_number','barcode']
X = df[feature_cols].fillna('').astype(str).agg(' | '.join, axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
features = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True)),
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(2,5), sublinear_tf=True))
])
model = Pipeline([
    ('features', features),
    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
])
model.fit(X_train, y_train)


## 3. Evaluate the model


In [ ]:
pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, pred))
print('Precision (suspicious):', precision_score(y_test, pred, pos_label='suspicious'))
print('Recall (suspicious):', recall_score(y_test, pred, pos_label='suspicious'))
print('F1 (suspicious):', f1_score(y_test, pred, pos_label='suspicious'))
print('\nClassification report:\n')
print(classification_report(y_test, pred))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))


## 4. Test a new record
The result below is a model prediction only. It is not proof of authenticity.


In [ ]:
new_record = {
    'drug_name': 'DEMO Paracetamol 500mg',
    'nafdac_number': 'A11-1234',
    'manufacturer': 'Demo Pharma Nigeria Ltd',
    'batch_number': 'B12345',
    'barcode': '1234567890123'
}
new_text = ' | '.join(str(new_record[c]) for c in feature_cols)
print('Prediction:', model.predict([new_text])[0])
print('Class probabilities:', dict(zip(model.classes_, model.predict_proba([new_text])[0])))


In [ ]:
joblib.dump(model, '../model.joblib')
print('Saved model to ../model.joblib')


## 5. Limitations and future work
- The dataset is synthetic and cannot establish real-world counterfeit detection performance.
- A real system should connect to an authoritative product registry instead of relying only on ML.
- Barcode scanning should use validated product identifiers and a trusted backend.
- Future work: validated NAFDAC data access, better explainability, drift monitoring, security testing, and user research.
